# Export QHARM Thermochemistry to Common CSV Tables

This notebook appends low-frequency-corrected thermochemistry from `BorylXAT-DB_qh_update.db` to the commonly reused CSV tables. Existing RRHO columns are preserved; QHARM columns are appended at the end.

Covered tables:

- `Data/csvs/reactants_B.csv`
- `Data/csvs/reactants_N.csv`
- `Data/csvs/reactants_Cl.csv`
- `Data/csvs/reactants_B_N.csv`
- `Data/csvs/reactants_B_N_full.csv`
- `Data/csvs/First_12000.csv`
- `Data/TS/Borane_all.csv`


In [ ]:
from __future__ import annotations

import json
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from DFTStructureGenerator.thermochemistry import (
    database_path,
    load_reaction_dataset,
    load_structure_thermochemistry,
)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'Data').exists():
    matches = [p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / 'Data').exists()]
    if not matches:
        raise FileNotFoundError('Run this notebook from the BorylXAT-DB repository tree.')
    REPO_ROOT = matches[0]

DB_PATH = database_path(True, REPO_ROOT)
HARTREE_TO_KCAL = 627.509474
LEGACY_HARTREE_TO_KCAL = 627.5
WRITE_IN_PLACE = True
CREATE_BACKUPS = True
BACKUP_DIR = REPO_ROOT / 'output' / 'qharm_csv_backups'
AUDIT_PATH = REPO_ROOT / 'output' / 'jupyter-notebook' / 'qharm_csv_export_audit.csv'

TARGETS = {
    'reactants_B': REPO_ROOT / 'Data' / 'csvs' / 'reactants_B.csv',
    'reactants_N': REPO_ROOT / 'Data' / 'csvs' / 'reactants_N.csv',
    'reactants_Cl': REPO_ROOT / 'Data' / 'csvs' / 'reactants_Cl.csv',
    'reactants_B_N': REPO_ROOT / 'Data' / 'csvs' / 'reactants_B_N.csv',
    'reactants_B_N_full': REPO_ROOT / 'Data' / 'csvs' / 'reactants_B_N_full.csv',
    'First_12000': REPO_ROOT / 'Data' / 'csvs' / 'First_12000.csv',
    'Borane_all': REPO_ROOT / 'Data' / 'TS' / 'Borane_all.csv',
}

for path in [DB_PATH, *TARGETS.values()]:
    if not path.exists():
        raise FileNotFoundError(path)

print('Database:', DB_PATH)
print('Write in place:', WRITE_IN_PLACE)
print('Backup directory:', BACKUP_DIR)


## Build QHARM lookup tables

All structure-level values come from `gibbs_qharm_hartree`. Derived quantities follow the original CSV definitions. Component-table `deltaG_react_qharm` remains in Hartree, while reaction-table values remain in kcal/mol, matching their existing companion columns.


In [ ]:
thermo = load_structure_thermochemistry(DB_PATH)

def keyed_thermo(category, pattern, id_columns, value_column):
    selected = thermo.loc[thermo['category'].eq(category), ['key', 'gibbs_qharm_hartree']].copy()
    identifiers = selected['key'].str.extract(pattern)
    if identifiers.isna().any().any():
        bad = selected.loc[identifiers.isna().any(axis=1), 'key'].head().tolist()
        raise ValueError(f'Could not parse {category} keys: {bad}')
    identifiers.columns = id_columns
    for column in id_columns:
        identifiers[column] = identifiers[column].astype(int)
    identifiers[value_column] = selected['gibbs_qharm_hartree'].to_numpy(dtype=float)
    if identifiers.duplicated(id_columns).any():
        raise ValueError(f'Duplicate QHARM keys for {category}: {id_columns}')
    return identifiers

b_qh = keyed_thermo('B', r'^B_(\d{5})$', ['Index'], 'G_energy_r_qharm')
lb_qh = keyed_thermo('LB', r'^LB_(\d{5})$', ['Index'], 'G_energy_r_qharm')
cl_r_qh = keyed_thermo('Cl', r'^Cl_(\d{5})_r$', ['Index'], 'G_energy_r_qharm')
cl_p_qh = keyed_thermo('c_radical', r'^Cl_(\d{5})_p$', ['Index'], 'G_energy_p_qharm')
bn_r_qh = keyed_thermo('complex_r', r'^B_(\d{5})_LB_(\d{5})_r$', ['B_Index', 'N_Index'], 'G_energy_r_qharm')
bn_p_qh = keyed_thermo('complex_p', r'^B_(\d{5})_LB_(\d{5})_p$', ['B_Index', 'N_Index'], 'G_energy_p_qharm')
ts_qh = keyed_thermo('ts', r'^B_(\d{5})_LB_(\d{5})_Cl_(\d{5})$', ['B_Index', 'N_Index', 'Cl_Index'], 'TS_G_qharm')

category_counts = thermo['category'].value_counts().rename_axis('category').rename('rows').to_frame()
display(category_counts)


In [ ]:
# Atomic chlorine has no vibrational correction. Recover its electronic/free-energy
# constant from both historical BDE definitions and require agreement.
cl_source = pd.read_csv(TARGETS['reactants_Cl'])
bn_source = pd.read_csv(TARGETS['reactants_B_N'])
cl_atom_estimates = pd.concat([
    cl_source['BDE_G'] / LEGACY_HARTREE_TO_KCAL - cl_source['deltaG_react'],
    bn_source['BDE_G'] / LEGACY_HARTREE_TO_KCAL + bn_source['deltaG_react'],
]).dropna()
G_CL_ATOM_HARTREE = float(cl_atom_estimates.median())
max_cl_atom_deviation = float((cl_atom_estimates - G_CL_ATOM_HARTREE).abs().max())
if max_cl_atom_deviation > 1e-6:
    raise ValueError(f'Inconsistent atomic-Cl reference: max deviation {max_cl_atom_deviation}')

cl_qh = cl_r_qh.merge(cl_p_qh, on='Index', validate='one_to_one')
cl_qh['deltaG_react_qharm'] = cl_qh['G_energy_p_qharm'] - cl_qh['G_energy_r_qharm']
cl_qh['BDE_G_qharm'] = (
    cl_qh['G_energy_p_qharm'] + G_CL_ATOM_HARTREE - cl_qh['G_energy_r_qharm']
) * HARTREE_TO_KCAL

bn_qh = bn_r_qh.merge(bn_p_qh, on=['B_Index', 'N_Index'], validate='one_to_one')
bn_qh = bn_qh.merge(
    b_qh.rename(columns={'Index': 'B_Index', 'G_energy_r_qharm': 'B_G_qharm'}),
    on='B_Index', validate='many_to_one',
).merge(
    lb_qh.rename(columns={'Index': 'N_Index', 'G_energy_r_qharm': 'N_G_qharm'}),
    on='N_Index', validate='many_to_one',
)
bn_qh['deltaG_comb_qharm(kcal/mol)'] = (
    bn_qh['G_energy_r_qharm'] - bn_qh['B_G_qharm'] - bn_qh['N_G_qharm']
) * HARTREE_TO_KCAL
bn_qh['deltaG_react_qharm'] = bn_qh['G_energy_p_qharm'] - bn_qh['G_energy_r_qharm']
bn_qh['BDE_G_qharm'] = (
    bn_qh['G_energy_r_qharm'] + G_CL_ATOM_HARTREE - bn_qh['G_energy_p_qharm']
) * HARTREE_TO_KCAL
bn_qh = bn_qh.drop(columns=['B_G_qharm', 'N_G_qharm'])

print('Atomic Cl reference (Hartree):', G_CL_ATOM_HARTREE)
print('BN QHARM rows:', len(bn_qh), 'Cl QHARM rows:', len(cl_qh))


## Enrich the CSV tables in memory

Repeated runs are idempotent: any existing QHARM columns are refreshed, and all original columns retain their order and values. `reactants_B_N_full.csv` is filled only for the selected B/LB coordination site represented in the QHARM database.


In [ ]:
def append_lookup_columns(source, lookup, keys, qharm_columns, *, validate='many_to_one'):
    original_columns = [c for c in source.columns if c not in qharm_columns]
    clean = source[original_columns].copy()
    result = clean.merge(
        lookup[keys + qharm_columns],
        on=keys, how='left', sort=False, validate=validate,
    )
    if len(result) != len(source):
        raise ValueError(f'Row count changed during merge on {keys}: {len(source)} -> {len(result)}')
    if not result[original_columns].equals(source[original_columns]):
        raise ValueError(f'Original columns changed during merge on {keys}')
    return result[original_columns + qharm_columns]

def enrich_reaction_table(source, *, include_ts):
    qharm_columns = ['G_energy_qharm', 'deltaG_react_qharm']
    if include_ts:
        qharm_columns += ['TS_G_qharm', 'deltaG_qharm(kcal/mol)', 'deltaGa_qharm(kcal/mol)']
    original_columns = [c for c in source.columns if c not in qharm_columns]
    clean = source[original_columns].copy()
    bn_part = bn_qh[[
        'B_Index', 'N_Index', 'G_energy_r_qharm', 'G_energy_p_qharm', 'deltaG_react_qharm'
    ]].rename(columns={
        'G_energy_r_qharm': '_bn_r_qharm',
        'G_energy_p_qharm': '_bn_p_qharm',
        'deltaG_react_qharm': '_bn_delta_qharm',
    })
    cl_part = cl_qh[[
        'Index', 'G_energy_r_qharm', 'G_energy_p_qharm', 'deltaG_react_qharm'
    ]].rename(columns={
        'Index': 'Cl_Index',
        'G_energy_r_qharm': '_cl_r_qharm',
        'G_energy_p_qharm': '_cl_p_qharm',
        'deltaG_react_qharm': '_cl_delta_qharm',
    })
    result = clean.merge(bn_part, on=['B_Index', 'N_Index'], how='left', sort=False, validate='many_to_one')
    result = result.merge(cl_part, on='Cl_Index', how='left', sort=False, validate='many_to_one')
    result['G_energy_qharm'] = result['_bn_r_qharm'] + result['_cl_r_qharm']
    result['deltaG_react_qharm'] = (result['_bn_delta_qharm'] + result['_cl_delta_qharm']) * HARTREE_TO_KCAL
    if include_ts:
        result = result.merge(ts_qh, on=['B_Index', 'N_Index', 'Cl_Index'], how='left', sort=False, validate='one_to_one')
        result['deltaG_qharm(kcal/mol)'] = (
            result['_bn_p_qharm'] + result['_cl_p_qharm'] - result['G_energy_qharm']
        ) * HARTREE_TO_KCAL
        result['deltaGa_qharm(kcal/mol)'] = (
            result['TS_G_qharm'] - result['G_energy_qharm']
        ) * HARTREE_TO_KCAL
    result = result.drop(columns=[c for c in result.columns if c.startswith('_')])
    if len(result) != len(source):
        raise ValueError(f'Reaction-table row count changed: {len(source)} -> {len(result)}')
    if not result[original_columns].equals(source[original_columns]):
        raise ValueError('Original reaction-table columns changed')
    return result[original_columns + qharm_columns]


In [ ]:
sources = {name: pd.read_csv(path) for name, path in TARGETS.items()}
enriched = {}
qharm_columns = {}

qharm_columns['reactants_B'] = ['G_energy_r_qharm']
enriched['reactants_B'] = append_lookup_columns(
    sources['reactants_B'], b_qh, ['Index'], qharm_columns['reactants_B'], validate='one_to_one'
)
qharm_columns['reactants_N'] = ['G_energy_r_qharm']
enriched['reactants_N'] = append_lookup_columns(
    sources['reactants_N'], lb_qh, ['Index'], qharm_columns['reactants_N']
)
qharm_columns['reactants_Cl'] = [
    'G_energy_r_qharm', 'G_energy_p_qharm', 'deltaG_react_qharm', 'BDE_G_qharm'
]
enriched['reactants_Cl'] = append_lookup_columns(
    sources['reactants_Cl'], cl_qh, ['Index'], qharm_columns['reactants_Cl'], validate='one_to_one'
)
qharm_columns['reactants_B_N'] = [
    'G_energy_r_qharm', 'G_energy_p_qharm', 'deltaG_comb_qharm(kcal/mol)',
    'deltaG_react_qharm', 'BDE_G_qharm',
]
enriched['reactants_B_N'] = append_lookup_columns(
    sources['reactants_B_N'], bn_qh, ['B_Index', 'N_Index'],
    qharm_columns['reactants_B_N'], validate='one_to_one',
)
selected_bn_sites = sources['reactants_B_N'][['B_Index', 'N_Index', 'N_Atomid']].merge(
    bn_qh, on=['B_Index', 'N_Index'], how='inner', validate='one_to_one'
)
qharm_columns['reactants_B_N_full'] = qharm_columns['reactants_B_N'].copy()
enriched['reactants_B_N_full'] = append_lookup_columns(
    sources['reactants_B_N_full'], selected_bn_sites, ['B_Index', 'N_Index', 'N_Atomid'],
    qharm_columns['reactants_B_N_full'], validate='many_to_one',
)
qharm_columns['First_12000'] = ['G_energy_qharm', 'deltaG_react_qharm']
enriched['First_12000'] = enrich_reaction_table(sources['First_12000'], include_ts=False)
qharm_columns['Borane_all'] = [
    'G_energy_qharm', 'deltaG_react_qharm', 'TS_G_qharm',
    'deltaG_qharm(kcal/mol)', 'deltaGa_qharm(kcal/mol)',
]
enriched['Borane_all'] = enrich_reaction_table(sources['Borane_all'], include_ts=True)

print('Prepared', len(enriched), 'CSV tables in memory.')


## Validate coverage and numerical consistency

The two unavailable Lewis bases (`Index` 224 and 327) and the two external chlorides (`Index` 723 and 724) have no rows in the QHARM database and therefore remain `NaN`. All retained B/LB complexes, the 12,000 attempted reactions, and the 9,237 final TS reactions must be complete.


In [ ]:
audit_rows = []
for name, frame in enriched.items():
    columns = qharm_columns[name]
    missing = frame[columns].isna().sum().to_dict()
    audit_rows.append({
        'table': name,
        'path': str(TARGETS[name].relative_to(REPO_ROOT)),
        'rows': len(frame),
        'qharm_complete_rows': int(frame[columns].notna().all(axis=1).sum()),
        'qharm_columns': ', '.join(columns),
        'missing_by_column': json.dumps(missing, sort_keys=True),
    })
audit_df = pd.DataFrame(audit_rows)

assert enriched['reactants_B']['G_energy_r_qharm'].notna().all()
n_valid = sources['reactants_N']['G_energy_r'].notna()
assert enriched['reactants_N'].loc[n_valid, 'G_energy_r_qharm'].notna().all()
cl_expected = ~sources['reactants_Cl']['Index'].isin([723, 724])
assert enriched['reactants_Cl'].loc[cl_expected, qharm_columns['reactants_Cl']].notna().all().all()
assert enriched['reactants_B_N'][qharm_columns['reactants_B_N']].notna().all().all()
assert enriched['reactants_B_N_full'][qharm_columns['reactants_B_N_full']].notna().all(axis=1).sum() == len(selected_bn_sites)
assert enriched['First_12000'][qharm_columns['First_12000']].notna().all().all()
assert enriched['Borane_all'][qharm_columns['Borane_all']].notna().all().all()

# Cross-check the independently derived reaction quantities against the values stored on DB TS rows.
db_reactions = load_reaction_dataset(TARGETS['Borane_all'], use_qharm=True, db_path=DB_PATH)
barrier_error = np.max(np.abs(
    enriched['Borane_all']['deltaGa_qharm(kcal/mol)'].to_numpy()
    - db_reactions['deltaGa(kcal/mol)'].to_numpy()
))
reaction_error = np.max(np.abs(
    enriched['Borane_all']['deltaG_qharm(kcal/mol)'].to_numpy()
    - db_reactions['deltaG(kcal/mol)'].to_numpy()
))
if barrier_error > 1e-8 or reaction_error > 1e-8:
    raise ValueError(f'DB cross-check failed: barrier={barrier_error}, reaction={reaction_error}')

display(audit_df)
print('Max barrier cross-check error:', barrier_error)
print('Max reaction-energy cross-check error:', reaction_error)


## Write validated tables

With `WRITE_IN_PLACE=True`, the notebook makes one untouched backup per source file under `output/qharm_csv_backups/`, writes through a temporary file, and then atomically replaces the source CSV. Set the switch to `False` for a validation-only run.


In [ ]:
AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
audit_df.to_csv(AUDIT_PATH, index=False)

if WRITE_IN_PLACE:
    if CREATE_BACKUPS:
        BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    for name, frame in enriched.items():
        destination = TARGETS[name]
        if CREATE_BACKUPS:
            backup = BACKUP_DIR / destination.relative_to(REPO_ROOT)
            backup.parent.mkdir(parents=True, exist_ok=True)
            if not backup.exists():
                shutil.copy2(destination, backup)
        temporary = destination.with_name(destination.name + '.qharm.tmp')
        frame.to_csv(temporary, index=False)
        os.replace(temporary, destination)
        print('Updated:', destination.relative_to(REPO_ROOT))
else:
    print('Validation-only run: no source CSV files were changed.')

print('Audit:', AUDIT_PATH.relative_to(REPO_ROOT))


## Result

The original RRHO fields remain unchanged. Downstream notebooks can select the appended QHARM fields explicitly, while `DFTStructureGenerator.thermochemistry.load_reaction_dataset()` continues to provide the same switch-based interface.
